## 第 10 课：无状态 RNG 与按需生成 mask

题目：[Triton: Low-Memory Dropout with Seeded RNG](https://www.deep-ml.com/problems/976?from=Triton%20Essentials)（ID 976）

计算目标（inverted dropout，概率 p）：

In [ ]:
draw = uniform_random(seed, i)          # i 是元素下标
output[i] = x[i] / (1 - p)   if draw > p
output[i] = 0                otherwise

核心约束：**不要**把 mask 物化（materialize）到内存里，而是在 kernel 内部用 seed 现场重算随机数。

例如（示意，p=0.5）：

In [ ]:
x = [1.0, 2.0, 3.0, 4.0]
随机数 = [0.1, 0.8, 0.6, 0.2]   # 只有 > 0.5 的保留

output = [0.0, 4.0, 6.0, 0.0]

保留的元素除以 `(1-p)=0.5` 后：2.0/0.5=4.0，3.0/0.5=6.0。

### 1. tl.rand：无状态、可复现

In [ ]:
rand = tl.rand(seed, offsets)

- `(seed, offsets)` 这个二元组**完全决定**随机数，kernel 内部没有任何生成器状态。
- 相同 seed + 相同 offsets → 完全相同的结果，所以题目要求"相同 seed 必须产出相同输出"是可复现的。

对比 PyTorch 的 `torch.rand`：那是维护内部状态（stateful）的生成器，而 Triton 的 `tl.rand` 是纯函数。

### 2. keep mask 与缩放一步到位

In [ ]:
keep = rand > p
output_value = tl.where(keep, x / (1 - p), 0.0)

### 3. 为什么叫 Low-Memory？

常规 dropout 需要把 mask 存下来（PyTorch 里是为了 backward 反向传播用）。这里只在 kernel 里按需重算：

- 省掉一块 mask 的全局内存；
- 省掉把 mask 写进内存、再读回来的带宽。

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def dropout_kernel(
    x_ptr,
    output_ptr,
    n_elements,
    p,
    seed,
    BLOCK_SIZE: tl.constexpr,
):
    # TODO 1：program ID / offsets / mask（和 Vector Add 一样的开场）

    # TODO 2：加载 x

    # TODO 3：rand = tl.rand(seed, offsets)
    # (seed, offsets) 完全决定随机数，无内部状态 → 相同 seed 可复现

    # TODO 4：keep = rand > p

    # TODO 5：output_value = tl.where(keep, x / (1 - p), 0.0)
    # 不物化 mask，直接在 kernel 里重算

    # TODO 6：带 mask 写入 output
    pass


def dropout(x: torch.Tensor, p: float, seed: int) -> torch.Tensor:
    BLOCK_SIZE = 1024

    # TODO 7：分配 output（形状、dtype 与 x 相同）

    # TODO 8：创建一维 grid

    # TODO 9：启动 kernel（p、seed 作为运行时参数传入）

    # TODO 10：返回 output
    pass

同时回答：

1. 为什么随机数用 `tl.rand(seed, offsets)` 在 kernel 里生成，而不是先在 Python 里 `torch.rand` 生成 mask 传进去？（提示：题目叫 Low-Memory Dropout）
2. 为什么相同 seed 一定得到相同输出？`tl.rand` 的 `(seed, offset)` 各起什么作用？
3. 为什么要除以 `(1 - p)`？如果不缩放，输出的期望会变成什么？

把代码和三个答案发给我，我继续审查。